# MedGraphRAG — MIMIC pilot (Gate A, self-hosted)

**Retrieval-mandatory clinical QA.** Companion to the MedQA POC; kept separate on purpose.

**Compliance:** generation runs on the Kaggle GPU via `LocalHFClient` — **no patient text leaves in an
API call**. Retrieval for this pilot is **BM25 (local, no embeddings)**. Dense/Graph arms come later,
once a *local* embedder is wired.

Pilot arms: **No-RAG** (closed book, + Gate-A feature capture) and **BM25**. The reward is
`r(q) = 1[BM25 correct] − 1[No-RAG correct]`; Phase A tests whether pre-retrieval uncertainty predicts it.

> Set `CFG.USE_SYNTHETIC = False` for the real run. With it `True`, the whole pipeline runs anywhere
> (fake model + synthetic data) to validate plumbing.

In [ ]:
# --- setup: make `mgr` importable + deps for LocalHFClient (torch is preinstalled on Kaggle GPU) ---
import os, sys, subprocess
REPO_URL = "https://github.com/SLIMIHamda/clinical_graphrag_study.git"  # TODO: confirm/replace
REPO_DIR = "clinical_graphrag_study"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=False)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)                      # both `mgr` and `manifest` live here
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=False)
print("python", sys.version.split()[0])

In [ ]:
# --- config ---
from types import SimpleNamespace
CFG = SimpleNamespace(
    BENCHMARK   = "MIMIC-QA",
    LOCAL_MODEL = "meta-llama/Llama-3.1-8B-Instruct",  # any local instruct model with a chat template
    N_ITEMS     = 200,          # pilot size; scale up once green
    SEEDS       = (42, 7, 123),
    GATE_N_SAMPLES = 5,         # self-consistency draws (bumped from MedQA's 2)
    CTX_TOP_K   = 8,            # retrieved chunks per question
    MAX_NEW_TOKENS = 8,         # MCQ answer is a single letter
    DATA_DIR    = "mimic_data",
    OUT_DIR     = "mimic_runs",
    ARMS        = ("No-RAG", "BM25"),   # both fully local; add Dense/Graph once a local embedder exists
    USE_SYNTHETIC = True,       # True = plumbing dry-run (no GPU/data); False = real run
)
os.makedirs(CFG.DATA_DIR, exist_ok=True); os.makedirs(CFG.OUT_DIR, exist_ok=True)
print(CFG)

## 1 · Data

Produce two things:
- **`qa_rows`** in the mgr MCQ schema: `{"qid","question","options":{"A":..,"B":..,"C":..,"D":..},"answer":"B"}`
- **`corpus_records`**: `{"id": chunk_id, "text": chunk}` — the patient note chunks that are the retrieval target.

The answer must live in the corpus, not in the model — that is what makes the task retrieval-mandatory.
For the real run, replace the synthetic block with your chosen credentialed MCQ set (e.g. EHRNoteQA over
MIMIC-IV discharge summaries): map each record to the schema above and chunk that patient's notes.

In [ ]:
# --- data prep ---
import json

def chunk_text(text, size=1200, overlap=150):
    text = " ".join((text or "").split())
    out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += max(1, size - overlap)
    return out

if CFG.USE_SYNTHETIC:
    # Tiny synthetic MIMIC-like set: exercises the whole pipeline before real data/GPU.
    qa_rows, corpus_records = [], []
    for i in range(CFG.N_ITEMS):
        cid = f"p{i:04d}_note0"
        dose = 10 + i % 5
        note = (f"Discharge summary, patient {i:04d}. History of present illness ... "
                f"The patient was prescribed drug-{i%7} at dose {dose} mg twice daily. Plan: follow-up.")
        corpus_records.append({"id": cid, "text": note})
        opts = {"A": f"{dose} mg", "B": f"{dose+10} mg", "C": f"{max(1,dose-5)} mg", "D": "not documented"}
        qa_rows.append({"qid": f"{i:04d}",
                        "question": f"What dose of drug-{i%7} did patient {i:04d} receive?",
                        "options": opts, "answer": "A",
                        "patient_id": f"{i:04d}", "note_ids": [cid]})
else:
    # TODO: load the real credentialed MCQ set here.
    #   for each item -> qa_rows.append({"qid","question","options"{A..D},"answer", "patient_id","note_ids"})
    #   for each patient note -> corpus_records += [{"id": f"{pid}_{k}", "text": ch} for k,ch in enumerate(chunk_text(note))]
    raise NotImplementedError("Plug in real MIMIC MCQ loading (see reports/mimic-design.html).")

from mgr.data.loader import write_items_fixture
write_items_fixture(CFG.BENCHMARK, CFG.DATA_DIR, qa_rows)   # writes {DATA_DIR}/MIMIC-QA.jsonl
print(f"{len(qa_rows)} QA items, {len(corpus_records)} corpus chunks")

In [ ]:
# --- retriever (BM25, local) + generation client (self-hosted) ---
from mgr.retrieval.bm25 import BM25Index, BM25Retriever
from mgr.retrieval.base import NullRetriever

BM25 = BM25Retriever(BM25Index.from_records(corpus_records, id_key="id", text_key="text"))

if CFG.USE_SYNTHETIC:
    class FakeLocalClient:            # plumbing only: fixed letter + canned logprobs
        model = "fake-local"
        def complete_text(self, model, messages, **p):
            return "A", {"in": 50, "out": 1}
        def complete_text_logprobs(self, model, messages, *, top_logprobs=10, **p):
            content = [{"token": "A", "logprob": -0.2, "top_logprobs": [
                {"token": "A", "logprob": -0.2}, {"token": "B", "logprob": -1.6},
                {"token": "C", "logprob": -2.3}, {"token": "D", "logprob": -3.0}]}]
            return "A", {"in": 100, "out": 1}, content
    GEN = FakeLocalClient()
else:
    from mgr.clients.local_hf import LocalHFClient
    GEN = LocalHFClient(model_id=CFG.LOCAL_MODEL, max_new_tokens=CFG.MAX_NEW_TOKENS)   # runs on the GPU, no API
print("generation client:", getattr(GEN, "model", type(GEN).__name__))

In [ ]:
# --- run the arms (No-RAG captures Gate-A features; BM25 gives the reward) ---
from types import SimpleNamespace
from mgr.generate.executor import RAGExecutor, GateCapture

def cfg_for():
    return {"benchmark": {"type": "MCQ", "n_items": CFG.N_ITEMS},
            "backbone": {"model_id": CFG.LOCAL_MODEL},
            "base": {"decoding": {"temperature": 0.0, "max_tokens": CFG.MAX_NEW_TOKENS}}}

items_by_arm = {}
for arm in CFG.ARMS:
    retriever = NullRetriever() if arm == "No-RAG" else BM25
    gate = GateCapture(enabled=True, n_samples=CFG.GATE_N_SAMPLES) if arm == "No-RAG" else None
    for seed in CFG.SEEDS:
        row = SimpleNamespace(retr_depth_k=CFG.CTX_TOP_K, seed=seed, benchmark=CFG.BENCHMARK)
        execu = RAGExecutor(client=GEN, data_root=CFG.DATA_DIR, retriever=retriever,
                            n_items=CFG.N_ITEMS, gate=gate)
        res = execu(row, cfg_for())
        path = f"{CFG.OUT_DIR}/{arm}_s{seed}.jsonl"
        with open(path, "w", encoding="utf-8") as fh:
            for it in res.items:
                fh.write(json.dumps(it) + "\n")
        items_by_arm.setdefault(arm, {})[seed] = res.items
        print(f"{arm:7s} s{seed}: acc={res.metrics['generation']['accuracy']:.3f}  n_err={res.metrics['generation']['n_item_errors']}  -> {path}")

In [ ]:
# --- GO/NO-GO: closed-book No-RAG must be LOW, else the task is not retrieval-mandatory ---
import statistics as st
norag_acc = st.mean(sum(it["correct"] for it in v) / len(v) for v in items_by_arm["No-RAG"].values())
print(f"No-RAG accuracy (mean over seeds): {norag_acc:.3f}   (MCQ chance ~0.25)")
print("OK: retrieval-mandatory" if norag_acc < 0.45 else
      "WARNING: No-RAG too high -> task may be parametric or leaked (you are back in the MedQA regime)")

In [ ]:
# --- Phase A: does pre-retrieval uncertainty predict the retrieval reward? ---
from mgr.analysis.gate_signal import gate_signal_analysis

def by_qid(items): return {it["qid"]: it for it in items}
norag_seeds = [by_qid(v) for v in items_by_arm["No-RAG"].values()]
bm25_seeds  = [by_qid(v) for v in items_by_arm["BM25"].values()]

DET = ("confidence", "entropy", "margin", "q_len_chars", "q_len_words", "n_options")
SC  = ("sc_agreement", "sc_entropy", "sc_matches_greedy")
records = []
for q in norag_seeds[0]:
    nr = [s[q] for s in norag_seeds if q in s]
    rg = [s[q] for s in bm25_seeds if q in s]
    if not nr or not rg:
        continue
    rec = {"qid": q}
    for k in DET: rec[k] = nr[0].get(k, 0.0)
    for k in SC:  rec[k] = round(sum(r.get(k, 0.0) for r in nr) / len(nr), 4)   # avg sc across seeds
    rec["no_rag_correct"] = int(sum(int(r["correct"]) for r in nr) * 2 >= len(nr))
    rec["rag_correct"]    = int(sum(int(r["correct"]) for r in rg) * 2 >= len(rg))
    records.append(rec)

report = gate_signal_analysis(records, lam=0.02)
with open(f"{CFG.OUT_DIR}/gate_signal.json", "w", encoding="utf-8") as fh:
    json.dump(report, fh, indent=2)
print(json.dumps(report, indent=2))

## Next

- **Real run:** set `USE_SYNTHETIC=False`, plug in the credentialed MCQ set + patient-note corpus, run on a
  GPU. Smoke-test one No-RAG item first (`confidence` in (0,1), `entropy` > 0, `sc_*` populated).
- **Add Dense/Graph:** wire a *local* embedder (no embeddings API) and extend `ARMS`; then the reward arm
  can move from BM25 to the hybrid.
- **Read the report:** high univariate AUROC + positive held-out `recovered_fraction` for the **logistic**
  gate = the premise transfers and is now powered. The `underpowered` flag clears once rescueable items
  number in the hundreds.
- The synthetic dry-run only checks plumbing — its `gate_signal` numbers are meaningless (the fake model
  ignores context, so No-RAG == BM25 and there is nothing to rescue).